# SymPy Verification of the Black-Scholes Formula

This notebook uses **SymPy** to verify the Black-Scholes results from two angles:

1. **Key identity** — $S\,e^{-qT}\,N'(d_1) = K\,e^{-rT}\,N'(d_2)$: verified symbolically.
2. **Greeks** — $\Delta, \Gamma, \nu, \Theta, \rho$: computed as symbolic derivatives, then cross-checked numerically.
3. **PDE residual** — verified numerically via finite differences.
4. **Put-call parity** — verified symbolically.

All symbolic results are cross-checked against `scipy.stats.norm` at reference parameter values
$S=100,\ K=100,\ T=1,\ r=5\%,\ q=2\%,\ \sigma=20\%$.

In [1]:
import sympy as sp
import numpy as np
from scipy.stats import norm

sp.init_printing(use_unicode=True)

# Reference parameters used for all numerical checks
REF = dict(S=100.0, K=100.0, T=1.0, r=0.05, q=0.02, sigma=0.20)

## 1. Symbols and Formula

We define:
$$d_1 = \frac{\ln(S/K) + (r-q+\sigma^2/2)\,T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$
$$N(x) = \tfrac{1}{2}\bigl(1 + \operatorname{erf}(x/\sqrt{2})\bigr), \qquad N'(x) = n(x) = \frac{e^{-x^2/2}}{\sqrt{2\pi}}$$
$$C = S\,e^{-qT}\,N(d_1) - K\,e^{-rT}\,N(d_2)$$

In [2]:
S, K, T, r, q, sigma = sp.symbols('S K T r q sigma', positive=True)

d1 = (sp.log(S/K) + (r - q + sigma**2/2) * T) / (sigma * sp.sqrt(T))
d2 = d1 - sigma * sp.sqrt(T)

def N_cdf(x):
    """Standard normal CDF via erf."""
    return sp.Rational(1, 2) * (1 + sp.erf(x / sp.sqrt(2)))

def n_pdf(x):
    """Standard normal PDF."""
    return sp.exp(-x**2 / 2) / sp.sqrt(2 * sp.pi)

# Call and put
C = S * sp.exp(-q*T) * N_cdf(d1) - K * sp.exp(-r*T) * N_cdf(d2)
P = K * sp.exp(-r*T) * N_cdf(-d2) - S * sp.exp(-q*T) * N_cdf(-d1)

# Helper: substitute REF params and evaluate numerically
def at_ref(expr):
    subs = {S: REF['S'], K: REF['K'], T: REF['T'],
            r: REF['r'], q: REF['q'], sigma: REF['sigma']}
    return float(expr.subs(subs).evalf())

print('d1 =')
display(d1)

print('d2 = d1 - sigma*sqrt(T) =')
display(d2)

print(f'\nC at reference params (SymPy): {at_ref(C):.4f}')

d1 =


  ⎛          2⎞         
  ⎜         σ ⎟      ⎛S⎞
T⋅⎜-q + r + ──⎟ + log⎜─⎟
  ⎝         2 ⎠      ⎝K⎠
────────────────────────
          √T⋅σ          

d2 = d1 - sigma*sqrt(T) =


          ⎛          2⎞         
          ⎜         σ ⎟      ⎛S⎞
        T⋅⎜-q + r + ──⎟ + log⎜─⎟
          ⎝         2 ⎠      ⎝K⎠
-√T⋅σ + ────────────────────────
                  √T⋅σ          


C at reference params (SymPy): 9.2270


## 2. Key Identity

$$S\,e^{-qT}\,n(d_1) = K\,e^{-rT}\,n(d_2)$$

**Proof sketch:** compute $d_1^2 - d_2^2 = (d_1 - d_2)(d_1 + d_2)$. Since $d_1 - d_2 = \sigma\sqrt{T}$:
$$d_1^2 - d_2^2 = \sigma\sqrt{T}\cdot\frac{2\ln(S/K) + 2(r-q)T}{\sigma\sqrt{T}} = 2\ln(S/K) + 2(r-q)T$$
Therefore:
$$\frac{n(d_1)}{n(d_2)} = e^{-(d_1^2-d_2^2)/2} = e^{-\ln(S/K)-(r-q)T} = \frac{K}{S}\,e^{-(r-q)T}$$
Rearranging gives the identity. SymPy verifies it symbolically:

In [3]:
lhs = S * sp.exp(-q*T) * n_pdf(d1)
rhs = K * sp.exp(-r*T) * n_pdf(d2)

diff_sym = sp.simplify(lhs - rhs)
print(f'sp.simplify(LHS - RHS) = {diff_sym}')
print(f'Identity holds symbolically: {diff_sym == 0}')

print(f'\nNumerical check at ref params:')
print(f'  LHS = {at_ref(lhs):.8f}')
print(f'  RHS = {at_ref(rhs):.8f}')

sp.simplify(LHS - RHS) = 0
Identity holds symbolically: True

Numerical check at ref params:
  LHS = 37.90115751
  RHS = 37.90115751


## 3. Greeks

We use the key identity above to simplify the derivatives. For each Greek:
- `sp.diff(C, x)` computes the raw symbolic derivative
- We evaluate it numerically at the reference params
- We compare to the closed-form formula evaluated with `scipy.stats.norm`

Expected formulas (call):

| Greek | Formula |
|-------|---------|
| $\Delta$ | $e^{-qT}\,N(d_1)$ |
| $\Gamma$ | $e^{-qT}\,n(d_1)\,/\,(S\sigma\sqrt{T})$ |
| $\nu$ (Vega) | $S\,e^{-qT}\,n(d_1)\,\sqrt{T}$ |
| $\Theta$ | $-\partial C/\partial T$ (see below) |
| $\rho$ | $K\,T\,e^{-rT}\,N(d_2)$ |

In [4]:
def bs_greeks_scipy(S, K, T, r, q, sigma):
    """Compute BS call Greeks using scipy.stats.norm.
    Parameter names match the REF dict so it can be called as bs_greeks_scipy(**REF).
    Inside the function these are plain floats — no conflict with the SymPy symbols above.
    """
    d1_v = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2_v = d1_v - sigma*np.sqrt(T)
    nd1, nd2, npd1 = norm.cdf(d1_v), norm.cdf(d2_v), norm.pdf(d1_v)
    eq, er = np.exp(-q*T), np.exp(-r*T)
    # dC/dT where T = time-to-maturity (positive: more time → higher call value)
    dC_dT = (S*eq*npd1*sigma/(2*np.sqrt(T))
             + r*K*er*nd2 - q*S*eq*nd1)
    return {
        'C':     S*eq*nd1 - K*er*nd2,
        'Delta': eq * nd1,
        'Gamma': eq * npd1 / (S * sigma * np.sqrt(T)),
        'Vega':  S * eq * npd1 * np.sqrt(T),
        'dC/dT': dC_dT,   # = +Theta_maturity (positive for calls)
        'Rho':   K * T * er * nd2,
    }

ref = bs_greeks_scipy(**REF)
print('Reference values (scipy):')
for k, v in ref.items():
    print(f'  {k:6s} = {v:.6f}')
print('\n  Note: market Theta = -dC/dT ≈', -ref["dC/dT"], '(negative = time decay)')

Reference values (scipy):
  C      = 9.227006
  Delta  = 0.586851
  Gamma  = 0.018951
  Vega   = 37.901158
  dC/dT  = 5.089319
  Rho    = 49.458109

  Note: market Theta = -dC/dT ≈ -5.0893189139983335 (negative = time decay)


In [5]:
greeks = [
    ('Delta', sp.diff(C, S),     'Delta'),
    ('Gamma', sp.diff(C, S, S),  'Gamma'),
    ('Vega',  sp.diff(C, sigma), 'Vega'),
    ('dC/dT', sp.diff(C, T),     'dC/dT'),   # T = time-to-maturity; positive for calls
    ('Rho',   sp.diff(C, r),     'Rho'),
]

print(f'{"Greek":6s}  {"SymPy":>12s}  {"scipy":>12s}  {"Match":>6s}')
print('-' * 46)
for name, expr, key in greeks:
    val_sympy = at_ref(expr)
    val_scipy = ref[key]
    match = np.isclose(val_sympy, val_scipy, rtol=1e-5)
    print(f'{name:6s}  {val_sympy:12.6f}  {val_scipy:12.6f}  {str(match):>6s}')

print('\nAll True: SymPy symbolic derivatives match the closed-form formulas.')

Greek          SymPy         scipy   Match
----------------------------------------------
Delta       0.586851      0.586851    True
Gamma       0.018951      0.018951    True
Vega       37.901158     37.901158    True
dC/dT       5.089319      5.089319    True
Rho        49.458109     49.458109    True

All True: SymPy symbolic derivatives match the closed-form formulas.


## 4. PDE Verification

The BS PDE in terms of $\tau = T$ (time-to-maturity) reads:
$$\frac{\partial C}{\partial \tau} = \frac{1}{2}\sigma^2 S^2\,\frac{\partial^2 C}{\partial S^2} + (r-q)S\,\frac{\partial C}{\partial S} - rC$$

i.e., the left-hand side (rate of change with increasing time-to-maturity) equals the right-hand side
(diffusion + drift − discount). We verify this numerically at the reference parameters
using centred finite differences.

In [6]:
def bs_call_np(S_v, K_v, T_v, r_v, q_v, sig_v):
    if T_v <= 0:
        return max(S_v - K_v, 0.0)
    d1_v = (np.log(S_v/K_v) + (r_v - q_v + 0.5*sig_v**2)*T_v) / (sig_v*np.sqrt(T_v))
    d2_v = d1_v - sig_v*np.sqrt(T_v)
    return S_v*np.exp(-q_v*T_v)*norm.cdf(d1_v) - K_v*np.exp(-r_v*T_v)*norm.cdf(d2_v)

S0 = REF['S']; K0 = REF['K']; T0 = REF['T']
r0 = REF['r']; q0 = REF['q']; s0 = REF['sigma']
h_S, h_T = 0.01, 1e-6

C0  = bs_call_np(S0,      K0, T0,      r0, q0, s0)
Cp  = bs_call_np(S0+h_S,  K0, T0,      r0, q0, s0)
Cm  = bs_call_np(S0-h_S,  K0, T0,      r0, q0, s0)
CTp = bs_call_np(S0,      K0, T0+h_T,  r0, q0, s0)
CTm = bs_call_np(S0,      K0, T0-h_T,  r0, q0, s0)

dC_dT   = (CTp - CTm) / (2*h_T)          # ≈ dC/dτ (positive)
dC_dS   = (Cp  - Cm)  / (2*h_S)          # ≈ Delta
d2C_dS2 = (Cp  - 2*C0 + Cm) / h_S**2    # ≈ Gamma

rhs = 0.5*s0**2*S0**2*d2C_dS2 + (r0-q0)*S0*dC_dS - r0*C0
residual = dC_dT - rhs

print('PDE: dC/dτ = ½σ²S²·Γ + (r-q)S·Δ − rC')
print(f'\n  dC/dτ         = {dC_dT:.6f}')
print(f'  ½σ²S²·Γ       = {0.5*s0**2*S0**2*d2C_dS2:.6f}')
print(f'  (r-q)S·Δ      = {(r0-q0)*S0*dC_dS:.6f}')
print(f'  -rC            = {-r0*C0:.6f}')
print(f'  RHS            = {rhs:.6f}')
print(f'\n  Residual dC/dτ - RHS = {residual:.3e}  (should be ≈ 0)')

PDE: dC/dτ = ½σ²S²·Γ + (r-q)S·Δ − rC

  dC/dτ         = 5.089319
  ½σ²S²·Γ       = 3.790116
  (r-q)S·Δ      = 1.760553
  -rC            = -0.461350
  RHS            = 5.089319

  Residual dC/dτ - RHS = 6.023e-08  (should be ≈ 0)


## 5. Put-Call Parity

$$C - P = S\,e^{-qT} - K\,e^{-rT}$$

This holds for any $N(x) + N(-x) = 1$, so SymPy should verify it purely from the erf identity.

In [7]:
parity_lhs = C - P
parity_rhs = S * sp.exp(-q*T) - K * sp.exp(-r*T)

diff_parity = sp.simplify(parity_lhs - parity_rhs)
print(f'C - P - (S·e^(-qT) - K·e^(-rT))  =  {diff_parity}')
print(f'Put-call parity verified: {diff_parity == 0}')

# Numerical sanity
parity_num  = at_ref(C) - at_ref(P)
forward_num = REF['S']*np.exp(-REF['q']*REF['T']) - REF['K']*np.exp(-REF['r']*REF['T'])
print(f'\nNumerical: C - P = {parity_num:.6f},  S·e^(-qT) - K·e^(-rT) = {forward_num:.6f}')

C - P - (S·e^(-qT) - K·e^(-rT))  =  0
Put-call parity verified: True

Numerical: C - P = 2.896925,  S·e^(-qT) - K·e^(-rT) = 2.896925
